# ChemBreak V8 Cloud
## Vertex AI + Colab Enterprise

This notebook uses the **ChemBreak_V8_Cloud** folder in your GitHub repository as its only ChemBreak codebase.

It does not import, read, or call V7 or any earlier ChemBreak version. You can later delete the V7 folder from GitHub without breaking V8.


## 1. Clone or refresh the GitHub repository

Upload the entire `ChemBreak_V8_Cloud` folder to your GitHub repository first. Then this notebook can clone the repository and run V8 from that folder.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os, shutil

REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
REPO_ROOT = Path("/tmp/ChemBreak_repo")
PROJECT_SUBDIR = "ChemBreak_V8_Cloud"

if (REPO_ROOT / ".git").exists():
    subprocess.run(["git","-C",str(REPO_ROOT),"pull","--ff-only"],check=True)
elif REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
    subprocess.run(["git","clone",REPO_URL,str(REPO_ROOT)],check=True)
else:
    subprocess.run(["git","clone",REPO_URL,str(REPO_ROOT)],check=True)

PROJECT_DIR = REPO_ROOT / PROJECT_SUBDIR
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"{PROJECT_SUBDIR} was not found in the repository. "
        "Upload the V8 Cloud folder to GitHub before continuing."
    )

PIPELINE = PROJECT_DIR / "scripts" / "chembreak_v8_cloud.py"
CONFIG_SOURCE = PROJECT_DIR / "config" / "run_config.json"

print("Repository:", REPO_ROOT)
print("V8 project:", PROJECT_DIR)
print("Pipeline:", PIPELINE)


## 2. Install the V8 requirements


In [ ]:
subprocess.run(
    [sys.executable,"-m","pip","install","-q","-r",str(PROJECT_DIR/"requirements.txt")],
    check=True
)
print("Requirements installed.")


## 3. Choose the run

Start with `test`. V8 writes into its own output namespace and never writes into V7 folders.

For pilot or production, add a Cloud Storage path if you have bucket permission.


In [ ]:
RUN_TYPE = "test"  # test, pilot, production
PROJECT_ID = "rs-foundsecft-mghasemi"
GCS_OUTPUT_URI = ""  # Example: gs://YOUR_BUCKET/ChemBreak/V8_Cloud/production

RUNTIME_DIR = Path("/tmp/ChemBreak_V8_Cloud_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG = RUNTIME_DIR / f"run_config_{RUN_TYPE}.json"

cfg = json.loads(CONFIG_SOURCE.read_text(encoding="utf-8"))
cfg["run_type"] = RUN_TYPE
cfg["project_id"] = PROJECT_ID
cfg["gcs_output_uri"] = GCS_OUTPUT_URI
RUNTIME_CONFIG.write_text(json.dumps(cfg,indent=2),encoding="utf-8")

OUTPUT_DIR = Path(f"/tmp/ChemBreak_V8_Cloud_Results/{RUN_TYPE}")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

def run_stage(stage):
    cmd = [
        sys.executable,"-u",str(PIPELINE),
        "--stage",stage,
        "--project-dir",str(PROJECT_DIR),
        "--config",str(RUNTIME_CONFIG),
        "--output-dir",str(OUTPUT_DIR),
    ]
    print(f"\n===== {stage.upper()} =====\n",flush=True)
    subprocess.run(cmd,check=True)

print("Run type:",RUN_TYPE)
print("Project ID:",PROJECT_ID)
print("Output:",OUTPUT_DIR)
print("Cloud persistence:",GCS_OUTPUT_URI or "OFF")


## 4. Preflight Vertex model access

This sends a harmless JSON test to the configured models. Do not begin task generation until the required roles report `OK`.


In [ ]:
run_stage("preflight")


## 5. Bootstrap fresh external source files

This downloads the upstream ChemSafety substance source and the upstream HarmBench behavior file directly. No old ChemBreak task bank is read.


In [ ]:
run_stage("bootstrap")


## 6. Build a fresh V8 assignment plan

The planner is part of V8 itself. It reads only V8 taxonomy and V8 source files.


In [ ]:
run_stage("plan")


## 7. Inspect the matrix coverage before generation


In [ ]:
import pandas as pd
from IPython.display import display

plan = pd.read_csv(OUTPUT_DIR/"assignments_v8.csv")
display(plan.head(20))
print("Assignments:",len(plan))
print("\nHC coverage")
display(plan["hc_id"].value_counts().sort_index())
print("\nHD coverage")
display(plan["hd_id"].value_counts().sort_index())
print("\nOT coverage")
display(plan["ot_id"].value_counts().sort_index())


## 8. Generate the candidate pool

By default, each assignment is sent to Gemini 3.1 Pro Preview, Llama 4 Maverick, and gpt-oss-120B. The purpose is candidate quality and diversity, not model comparison.


In [ ]:
run_stage("generate")


## 9. Deterministic validation


In [ ]:
run_stage("validate")


## 10. Repair invalid candidates


In [ ]:
run_stage("repair")


## 11. Blind judging and selection


In [ ]:
run_stage("judge")


## 12. Refill unresolved assignments, then judge the refills


In [ ]:
run_stage("refill")
run_stage("judge")


## 13. Finalize the current bank and inspect reports


In [ ]:
run_stage("finalize")
run_stage("status")

for name in ["run_summary.json","coverage_report.csv","diversity_report.csv","final_task_bank.csv"]:
    p = OUTPUT_DIR/name
    print("\n",name)
    if p.suffix==".json" and p.exists():
        print(p.read_text(encoding="utf-8"))
    elif p.exists():
        display(pd.read_csv(p).head(25))


## 14. Create a downloadable checkpoint ZIP


In [ ]:
summary = json.loads((OUTPUT_DIR/"run_summary.json").read_text(encoding="utf-8"))
label = summary["completion_label"]
archive = shutil.make_archive(
    f"/tmp/ChemBreak_V8_Cloud_{RUN_TYPE}_{label}",
    "zip",
    root_dir=str(OUTPUT_DIR)
)
print("Archive:",archive)
